---
title: "Data Cleaning"
format:
    html: 
        code-fold: true
---

<!-- After digesting the instructions, you can delete this cell, these are assignment instructions and do not need to be included in your final submission.  -->

{{< include introduction.qmd >}} 

{{< include methods.qmd >}} 

# Code 


# Step 1 — Load and Preview the Raw Data

## 1.1 Imports and file paths

I use DuckDB to read the JSONL file because it infers a schema over the nested fields and keeps a persistent copy of the raw table. pandas and NumPy handle the cleaning after that.

In [4]:
import duckdb          # reads the JSONL file and stores tables in a local database
import pandas as pd    # main cleaning work
import numpy as np     # NaN handling and np.select

from pathlib import Path

pd.set_option("display.max_columns", None)   # show every column in previews

In [5]:
# Paths are relative to this page's folder in the Quarto project
RAW_PATH = Path("../../data/raw-data/ravelry_patterns.jsonl")        # raw API pull, one pattern per line
DB_PATH = Path("../../data/processed-data/ravelry.duckdb")           # persistent DuckDB database
RATES_PATH = Path("../../data/raw-data/ecb_rates.csv")               # ECB exchange rates (Step 5)

## 1.2 Load the raw patterns into DuckDB

`sample_size = -1` makes DuckDB scan every record before inferring column types. Some fields are null in most records, so a partial sample could give them the wrong type.

In [6]:
def load_raw_patterns(con):
    """(Re)create the raw_patterns table from the JSONL file."""
    con.execute(f"""
                CREATE OR REPLACE TABLE raw_patterns AS
                SELECT * FROM read_json('{RAW_PATH}',
                                        format = 'newline_delimited',
                                        sample_size = -1);
    """)

# Open (or create) the database file, load the table, and confirm the row count
con = duckdb.connect(str(DB_PATH))
load_raw_patterns(con)
print(con.execute("SELECT COUNT(*) AS n_patterns FROM raw_patterns").fetchone())

(30038,)


## 1.3 Inspect the schema

`DESCRIBE` lists every column and the type DuckDB inferred. Most columns are plain values. The nested ones (`craft`, `pattern_type`, `pattern_author`, `pattern_categories`, and so on) come in as `STRUCT`s or lists of structs, which Step 8 flattens. A few columns are typed `"NULL"`, meaning they had no values in any record.

In [7]:
# One row per column: name, inferred DuckDB type, nullability
con.execute("""
    SELECT *
    FROM (DESCRIBE raw_patterns)
""").df()

,column_name,column_type,null,key,default,extra
0,comments_count,BIGINT,YES,None,None,None
1,created_at,VARCHAR,YES,None,None,None
2,currency,VARCHAR,YES,None,None,None
3,difficulty_average,DOUBLE,YES,None,None,None
4,difficulty_count,BIGINT,YES,None,None,None
5,downloadable,BOOLEAN,YES,None,None,None
6,favorites_count,BIGINT,YES,None,None,None
7,free,BOOLEAN,YES,None,None,None
8,gauge,DOUBLE,YES,None,None,None
9,gauge_divisor,BIGINT,YES,None,None,None


## 1.4 Pull into pandas and check missing values

From here on, the cleaning happens in pandas on `raw_df`. These missing-value counts decide which columns get dropped in Step 2. They undercount in text columns, though: Ravelry stores empty text fields as blank strings, which pandas doesn't treat as missing. Step 4 fixes that.

In [8]:
# Materialize the full raw table as a DataFrame
raw_df = con.execute("SELECT * FROM raw_patterns").df()
raw_df.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(30038, 54)

In [9]:
# Count of missing values per column
raw_df.isna().sum()

comments_count                 0
created_at                     0
currency                    3996
difficulty_average             0
difficulty_count           22412
downloadable                   1
favorites_count                0
free                           1
gauge                      11465
gauge_divisor               4194
gauge_pattern                144
gauge_repeats              29883
generally_available         1303
has_uk_terminology         18149
has_us_terminology         12880
id                             0
name                           0
pdf_url                     5978
permalink                      0
price                      15451
projects_count                 0
published                   5374
queued_projects_count          0
rating_average                 0
rating_count               22460
row_gauge                  14127
updated_at                     0
url                          169
yardage                    10966
yardage_max                14168
yarn_list_

# Step 2 — Structural Cleanup: Empty Columns and Duplicates

Four columns are missing in more than 90% of records. Three of them (`personal_attributes`, `unlisted_product_ids`, `volumes_in_library`) are empty in every row. According to the API docs they describe the logged-in API user (their notebook, their library, their editing permissions), not the pattern, so they come back empty for a data pull like this one. `gauge_repeats` is filled in for only about 150 patterns. None of them carry usable information, so I drop them.

Then I make sure each pattern `id` appears only once, in case the collection script pulled the same pattern twice.

In [10]:
# Columns with >90% missing values (see Step 1.4)
raw_df = raw_df.drop(columns=["gauge_repeats","personal_attributes","volumes_in_library","unlisted_product_ids"])


# Remove repeated pattern ids, keeping the first copy
n_before = len(raw_df)
raw_df = raw_df.drop_duplicates(subset="id", keep="first").copy()
print(f"Removed {n_before - len(raw_df):,} duplicate ids ({len(raw_df):,} patterns left)")

Removed 0 duplicate ids (30,038 patterns left)


# Step 3 — Scope: Keep Knitting and Crochet

## 3.1 Extract the craft name and check the terminology flags

`craft` is a nested struct, so I pull out just its `name`. Then I cross-tab the UK/US terminology flags against craft to see whether they're useful. Per the API docs, the flags only apply to crochet, where UK and US stitch names differ.

In [11]:
# Flatten the craft struct to its name ("Knitting", "Crochet", ...)
raw_df["craft_name"] = raw_df["craft"].str.get("name")

# Make the missing values visible as their own category in the table
uk = raw_df["has_uk_terminology"].astype("string").fillna("missing")
us = raw_df["has_us_terminology"].astype("string").fillna("missing")

# Terminology flag combinations by craft
pd.crosstab([uk, us], raw_df["craft_name"])

craft_name                             Crochet  Knitting  Loom Knitting  \
has_uk_terminology has_us_terminology                                     
False              False                  1935      6731             48   
                   True                   2115         3              0   
                   missing                   2         0              0   
True               False                   171         1              0   
                   True                    525         0              0   
                   missing                 312         1              0   
missing            False                     7         0              0   
                   True                   5571         6              0   
                   missing                1533     10899             74   

craft_name                             Machine Knitting  
has_uk_terminology has_us_terminology                    
False              False                             45  
                   True                               0  
                   missing                            0  
True               False                              0  
                   True                               0  
                   missing                            0  
missing            False                              0  
                   True                               0  
                   missing                           59

## 3.2 Drop the terminology columns

The flags are mostly missing, inconsistent (some crochet patterns are flagged both UK and US), and not part of my research question, so I drop them.

In [12]:
raw_df = raw_df.drop(columns=["has_uk_terminology","has_us_terminology"])

## 3.3 Keep only knitting and crochet

Loom knitting and machine knitting have only about a hundred patterns each. That's too few to compare against the two main crafts, so I keep only knitting and crochet.

In [13]:
KEEP_CRAFTS = ["Knitting", "Crochet"]


# Craft counts, then filter to the two crafts being analyzed
print(raw_df["craft_name"].value_counts(dropna=False).to_string())
raw_df = raw_df[raw_df["craft_name"].isin(KEEP_CRAFTS)].copy()

raw_df.shape

craft_name
Knitting            17641
Crochet             12171
Loom Knitting         122
Machine Knitting      104


(29812, 49)

# Step 4 — Placeholder Values

Two kinds of placeholders make missing data look like real data. I fix both before anything else uses these columns.

## 4.1 Blank strings in text columns

When a designer leaves a text field empty, Ravelry returns `""` instead of null. pandas counts `""` as a real value, so it's skipped by `isna()` and shows up as its own category in `value_counts()`. The table below compares truly missing values with blanks for every text column. Nested struct and list columns are also stored as `object`, so I keep only the columns whose values are all strings.

In [14]:
# Text columns only: nested struct/list columns are also "object" dtype, so keep columns whose values are all strings
text_cols = [c for c in raw_df.columns
             if raw_df[c].notna().any() and raw_df[c].dropna().map(lambda v: isinstance(v, str)).all()]

empty_summary = pd.DataFrame({
    "missing": raw_df[text_cols].isna().sum(),
    "blank": [(raw_df[c].str.strip() == "").sum() for c in text_cols],
}, index=text_cols)
empty_summary["total_empty"] = empty_summary["missing"] + empty_summary["blank"]
empty_summary["pct_empty"] = (empty_summary["total_empty"] / len(raw_df) * 100).round(1)

empty_summary.sort_values("total_empty", ascending=False)

,missing,blank,total_empty,pct_empty
pdf_url,5923,23178,29101,97.6
url,168,17568,17736,59.5
gauge_pattern,144,15020,15164,50.9
gauge_description,143,10058,10201,34.2
sizes_available,35,8352,8387,28.1
currency_symbol,5720,20,5740,19.3
currency,3971,1749,5720,19.2
published,5346,0,5346,17.9
notes,36,3286,3322,11.1
yarn_weight_description,2196,0,2196,7.4


The blanks are all in free-text fields (`gauge_pattern`, `gauge_description`, `sizes_available`, `notes`, `currency`). The structured columns only have real missing values. I convert blank and whitespace-only strings to missing and store the text columns as pandas' string type, so every missing value looks the same. This also trims stray spaces around real values so that `" Lace "` and `"Lace"` count as the same value.

In [15]:
# Blank or whitespace-only strings are missing values, not real categories.
# Converting to pandas' string dtype also makes every missing value the same <NA> (no mix of None/NaN)
for c in text_cols:
    raw_df[c] = raw_df[c].str.strip().replace("", pd.NA).astype("string")

# Confirm: no blanks left
print(sum((raw_df[c].str.strip() == "").sum() for c in text_cols), "blanks remaining")

0 blanks remaining


## 4.2 Ratings and difficulty: 0 votes means missing

A pattern no one has rated shows `rating_average = 0` with a missing `rating_count`. The same happens for difficulty. A 0 here means "no votes," not a real score: ratings run from 1 to 5, so leaving the zeros in would drag every average down. The check below confirms that the zero averages line up exactly with the missing counts.

In [16]:
# Number of zero averages vs. number of missing rating counts: these should match
print((raw_df["rating_average"] == 0).sum(), raw_df["rating_count"].isna().sum())

22243 22243


I apply the same rule to both rating and difficulty with one helper, so they're handled the same way. After this, each average is either a real score or `NaN`, and each count is an integer.

In [17]:
def zero_votes_to_missing(df, avg_col, count_col):
    """Set avg_col to NaN where count_col is 0 or missing; fill count_col with 0 as int."""
    df = df.copy()
    no_votes = df[count_col].isna() | (df[count_col] == 0)
    df.loc[no_votes, avg_col] = np.nan
    df[count_col] = df[count_col].fillna(0).astype(int)
    return df

raw_df = zero_votes_to_missing(raw_df, "rating_average", "rating_count")
raw_df = zero_votes_to_missing(raw_df, "difficulty_average", "difficulty_count")

# Check: the minimums should now be real scores, not 0
raw_df[["rating_average", "rating_count", "difficulty_average", "difficulty_count"]].describe()

,rating_average,rating_count,difficulty_average,difficulty_count
count,7569.000000,29812.000000,7618.000000,29812.000000
mean,4.536520,5.047363,3.025213,5.160539
std,0.425246,33.033643,1.152575,33.937007
min,1.666667,0.000000,1.000000,0.000000
25%,4.333333,0.000000,2.181818,0.000000
50%,4.645570,0.000000,2.818182,0.000000
75%,4.857143,3.000000,3.666667,3.000000
max,5.000000,2345.000000,8.500000,2329.000000


# Step 5 — Price and Availability

## 5.1 Why is `price` missing for paid patterns?

`price` is missing for about half the patterns. For free patterns that makes sense. But many patterns marked `free == False` also have no price. Checking them against `downloadable` shows most of these paid-but-unpriced patterns aren't downloadable at all. They're print-only patterns from books and magazines, which don't have a per-pattern price.

In [18]:
print(raw_df[["name", "free", "price", "downloadable"]].head(3), "\n")

# Is price missing, by free flag?
print("Price missing vs. free flag:")
print(pd.crosstab(raw_df["price"].isna(), raw_df["free"], dropna=False), "\n")

# Paid patterns with no price: how are they distributed?
paid_no_price = (raw_df["free"] == False) & raw_df["price"].isna()
print(f"Paid patterns with no price: {paid_no_price.sum():,}\n")

print("Download location type (paid, no price):")
print(raw_df.loc[paid_no_price, "download_location"].str.get("type").value_counts(dropna=False).to_string(), "\n")

print("Downloadable? (paid, no price):")
print(raw_df.loc[paid_no_price, "downloadable"].value_counts(dropna=False).to_string())

                       name   free  price  downloadable
0  Glassbrikke àla Mon Ami!  False   20.0          True
1                  Ma'alaea  False    9.5          True
2            Cirque du Sole  False    NaN         False 

Price missing vs. free flag:
free   False  True   <NA> 
price                     
False  13695    806      1
True    8368   6942      0 

Paid patterns with no price: 8,368

Download location type (paid, no price):
download_location
<NA>        8122
external     232
ravelry       14 

Downloadable? (paid, no price):
downloadable
False    8035
True      333


## 5.2 Build an `availability` category

Rather than imputing prices, I combine `free` and `downloadable` into one three-level variable that describes how someone can get the pattern. Per the API docs, `free` covers patterns free on Ravelry or elsewhere, and `downloadable` means the pattern can be downloaded digitally anywhere, not only through Ravelry.

| availability | meaning |
|---|---|
| `free` | free pattern |
| `paid_download` | paid, digital download (on Ravelry or an external site) |
| `print_only` | paid, available only in print (book or magazine) |

Rows where `free` itself is missing get dropped, since they can't be classified.

In [19]:
# Drop rows where the free flag is unknown
raw_df = raw_df[raw_df["free"].notna()]

is_free = raw_df["free"].fillna(False)
is_download = raw_df["downloadable"].fillna(False)

# Conditions are checked in order: free first, then downloadable, otherwise print_only
raw_df["availability"] = np.select(
    [is_free, is_download],
    ["free", "paid_download"],
    default="print_only",
)
print(raw_df["availability"].value_counts().to_string(), "\n")

raw_df[["name", "free", "price", "downloadable", "availability"]].groupby("availability").head(3)

availability
paid_download    13785
print_only        8278
free              7748 



,name,free,price,downloadable,availability
0,Glassbrikke àla Mon Ami!,False,20.0,True,paid_download
1,Ma'alaea,False,9.5,True,paid_download
2,Cirque du Sole,False,NaN,False,print_only
3,Yo-Yo Scarf,True,NaN,True,free
4,Ruffles and Pearls Sweater,False,10.0,True,paid_download
6,Wizard Warmers,True,NaN,True,free
10,Sea Glass Hat,True,NaN,True,free
11,Whirls and Turns - Motifs that Matter,False,NaN,False,print_only
16,Garnet / January,False,NaN,False,print_only


## 5.3 Where is the pattern sold?

`download_location` is a struct whose `type` is either `"ravelry"` (sold or hosted on Ravelry) or `"external"` (the designer's own site, a blog, a shop). Being on Ravelry itself could plausibly affect how visible a pattern is, so I keep the type as its own column.

In [20]:
# "ravelry", "external", or missing
raw_df["download_type"] = raw_df["download_location"].str.get("type")

# Is download_type missing mostly for print-only patterns?
pd.crosstab(raw_df["availability"], raw_df["download_type"].fillna("missing"))

download_type,external,missing,ravelry
availability,,,
free,4267,367,3114
paid_download,2143,534,11108
print_only,0,8269,9


Print-only patterns almost never have a download location, which matches the availability categories. Paid downloads are mostly sold on Ravelry, while free patterns are split between Ravelry and external sites. A small number of rows disagree (a few print-only patterns with a Ravelry download, and some free or paid downloads with no location). They're under 3% of patterns, so I leave them as recorded.

## 5.4 Convert prices to USD

Prices are listed in each designer's own currency, so a price of 6.00 could be USD, EUR, or DKK. To compare them, I convert everything to USD.

In [21]:
# Currencies among patterns that actually have a price
has_price = raw_df["price"].notna()
print(raw_df.loc[has_price, "currency"].value_counts(dropna=False).to_string())

currency
USD     9304
EUR     1947
GBP     1339
CAD      738
NOK      340
AUD      239
DKK      176
SEK      132
<NA>      95
CHF       38
JPY       35
NZD       33
PLN       31
RUB       14
BRL       13
SGD        9
CZK        6
MXN        6
HUF        1
INR        1
HRK        1
ZAR        1
HKD        1
TWD        1


I use the European Central Bank's daily reference rates on the collection date, so every price is converted as of the same day. The ECB file is downloaded once and saved, so rendering this page doesn't depend on the ECB site. Croatian kuna uses its fixed euro conversion rate, since Croatia adopted the euro in 2023. The original `price` and `currency` columns stay, and `price_usd` is added alongside them.

In [22]:
# ECB reference rates: units of each currency per 1 EUR, one row per business day
RATE_DATE = "2026-11-23"   # matches the collection date found in Step 6

# Download once and save, so rendering the page doesn't depend on the ECB site
if not RATES_PATH.exists():
    ecb = pd.read_csv("https://www.ecb.europa.eu/stats/eurofxref/eurofxref-hist.zip", compression="zip")
    ecb.to_csv(RATES_PATH, index=False)

ecb = pd.read_csv(RATES_PATH, parse_dates=["Date"])
day = ecb[ecb["Date"] <= RATE_DATE].sort_values("Date").iloc[-1]   # last business day on/before collection
eur_per = day.drop("Date").dropna().astype(float)                    # currency -> units per EUR
eur_per["EUR"] = 1.0                                                  # EUR isn't a column in the ECB file
eur_per["HRK"] = 7.53450                                              # kuna was replaced by the euro at this fixed rate (Jan 2023)

print(f"Rates from {day['Date'].date()}")

# price in currency -> EUR -> USD
rate = raw_df["currency"].map(eur_per)
raw_df["price_usd"] = (raw_df["price"] / rate * eur_per["USD"]).round(2)

# Anything priced but not converted (unknown currency or missing currency)
unconverted = has_price & raw_df["price_usd"].isna()
print(f"{unconverted.sum():,} priced patterns could not be converted")
print(raw_df.loc[unconverted, "currency"].value_counts(dropna=False).to_string())

raw_df[["name", "price", "currency", "price_usd"]].dropna().head()

Rates from 2026-11-23
110 priced patterns could not be converted
currency
<NA>    95
RUB     14
TWD      1


,name,price,currency,price_usd
0,Glassbrikke àla Mon Ami!,20.00,NOK,2.04
1,Ma'alaea,9.50,USD,9.5
4,Ruffles and Pearls Sweater,10.00,USD,10.0
5,Kinetic Cowl,7.99,USD,7.99
7,Pining Around Throw,3.99,USD,3.99


About 110 priced patterns (under 1%) can't be converted, because their currency is missing or isn't published by the ECB (Russian ruble, Taiwan dollar). Their `price_usd` is left missing.

# Step 6 — Dates and Time on Ravelry

## 6.1 Parse dates and measure time on Ravelry

Favorites and projects only build up while a pattern is on Ravelry, so the right exposure measure is time since the pattern was **added to Ravelry** (`created_at`), not since it was originally published. This matters for digitized vintage patterns: a 1940s pamphlet uploaded in 2015 has had about 11 years of exposure, not 80+.

I keep three dates, each meaning something different:

| column | source | meaning |
|---|---|---|
| `created_date` | `created_at` | when the pattern page was added to Ravelry (never missing) |
| `published_date` | `published` | original publication date shown on the pattern page; optional, can be decades earlier |
| `available_date` | `generally_available` | Ravelry's best estimate of when the pattern first became available to the public |

Missing `published_date` and `available_date` values are left as `NaT` rather than filled from `created_at`, because filling them would blur the difference between the dates. Time is measured to the latest `updated_at` in the data, which stands in for the collection date.

In [23]:
# Ravelry dates look like "2015/03/01" or "2015/03/01 10:00:00 -0400"; keep only the date part
created = raw_df["created_at"].astype("string").str.slice(0, 10).str.replace("/", "-")
published = raw_df["published"].astype("string").str.slice(0, 10).str.replace("/", "-")
available = raw_df["generally_available"].astype("string").str.slice(0, 10).str.replace("/", "-")
updated = raw_df["updated_at"].astype("string").str.slice(0, 10).str.replace("/", "-")

raw_df["created_date"] = pd.to_datetime(created, errors="coerce")       # added to Ravelry
raw_df["published_date"] = pd.to_datetime(published, errors="coerce")   # original publication (can be decades earlier)
raw_df["available_date"] = pd.to_datetime(available, errors="coerce")   # Ravelry's estimate of public availability
collected_on = pd.to_datetime(updated, errors="coerce").max()            # proxy for collection date

raw_df["years_on_ravelry"] = (collected_on - raw_df["created_date"]).dt.days / 365.25

print(f"Collection date used: {collected_on.date()}")
raw_df[["created_date", "published_date", "available_date", "years_on_ravelry"]].describe()

Collection date used: 2026-11-23


,created_date,published_date,available_date,years_on_ravelry
count,29811,24466,28522,29811.000000
mean,2016-06-17 21:34:24.647277,2015-01-02 20:22:03.109621,2013-12-13 06:21:41.058831,10.431488
min,2007-02-07 00:00:00,1901-01-01 00:00:00,1900-07-01 00:00:00,3.121150
25%,2012-11-18 12:00:00,2013-01-01 00:00:00,2011-12-08 00:00:00,6.543463
50%,2016-10-29 00:00:00,2016-12-01 00:00:00,2016-03-09 12:00:00,10.067077
75%,2020-05-08 00:00:00,2020-05-01 00:00:00,2020-01-19 00:00:00,14.010951
max,2023-10-10 00:00:00,2026-11-01 00:00:00,2026-11-01 00:00:00,19.791923
std,NaN,NaN,NaN,4.521070


`published_date` goes back to 1901. I checked the early dates: about 195 patterns are from before 1950, mostly the 1930s–40s, spread across many years with no single placeholder value. They're digitized vintage patterns, not errors. Many fall on January 1, which just means only the year was recorded. Since exposure is measured from `created_date`, which starts in 2007 when Ravelry launched, these old dates don't affect `years_on_ravelry`.

## 6.2 Sanity check: no patterns from the future

A pattern can't be added to Ravelry after the data was collected. Any row with negative time on Ravelry would have a bad date, so its date fields would be set to missing rather than the row being dropped.

In [24]:
# Negative = added after collection -> bad date
bad = raw_df["years_on_ravelry"] < 0
print(f"{bad.sum():,} patterns added after the collection date -> set to missing")
raw_df.loc[bad, ["years_on_ravelry", "created_date"]] = np.nan

0 patterns added after the collection date -> set to missing


# Step 7 — Gauge

Gauge is how many stitches (and rows) fit in a given width, and it mostly reflects yarn weight and needle size. Ravelry stores it in pieces:

| column | meaning |
|---|---|
| `gauge` | number of stitches |
| `gauge_divisor` | over how many inches: 1, 2, or 4 (per the API docs) |
| `row_gauge` | number of rows over the same width |
| `gauge_pattern` | the stitch the gauge is measured in (free text) |
| `gauge_description` | Ravelry's text version, e.g. "22 stitches and 30 rows = 4 inches in stockinette" |

So "22 stitches over 4 inches" and "5.5 stitches over 1 inch" are the same gauge stored differently.

## 7.1 Inspect how the pieces line up

In [25]:
gauge_cols = ["gauge", "gauge_divisor", "row_gauge", "gauge_pattern", "gauge_description"]

# How do the pieces line up?
print("Divisor values (where gauge is present):")
print(raw_df.loc[raw_df["gauge"].notna(), "gauge_divisor"].value_counts(dropna=False).to_string(), "\n")

print("Gauge present but divisor missing:", (raw_df["gauge"].notna() & raw_df["gauge_divisor"].isna()).sum())
print("Row gauge present but stitch gauge missing:", (raw_df["row_gauge"].notna() & raw_df["gauge"].isna()).sum(), "\n")

print(raw_df[["gauge", "row_gauge"]].describe(), "\n")

raw_df.loc[raw_df["gauge"].notna(), gauge_cols].sample(10, random_state=0)

Divisor values (where gauge is present):
gauge_divisor
4       14831
1        2421
2        1086
<NA>      117 

Gauge present but divisor missing: 117
Row gauge present but stitch gauge missing: 69 

              gauge    row_gauge
count  18455.000000  15801.00000
mean      17.212636     22.08248
std        8.460468     11.88717
min        0.000000      0.00000
25%       11.000000     12.00000
50%       18.000000     24.00000
75%       22.000000     30.00000
max       99.990000     99.99000 



,gauge,gauge_divisor,row_gauge,gauge_pattern,gauge_description
16082,13.0,4,9.5,trebble,13 stitches and 9.5 rows = 4 inches in trebble
29367,22.0,4,30.0,stockinette stitch on larger needles,22 stitches and 30 rows = 4 inches in stockine...
12175,26.0,4,34.0,<NA>,26 stitches and 34 rows = 4 inches
17213,6.0,1,NaN,st st,6 stitches = 1 inch in st st
917,5.0,2,4.0,<NA>,5 stitches and 4 rows = 2 inches
21705,20.0,4,24.0,Lace pattern,20 stitches and 24 rows = 4 inches in Lace pat...
24725,32.0,4,48.0,Stockinette Stitch,32 stitches and 48 rows = 4 inches in Stockine...
10941,4.0,2,5.0,"pre-felt:10sc x 10 rows = 4.25"" x 3.5"" felted ...",4 stitches and 5 rows = 2 inches in pre-felt:1...
25230,5.0,1,5.0,single crochet stitches,5 stitches and 5 rows = 1 inch in single croch...
3161,22.0,4,30.0,After 3 rounds 2.5 inch in diameter,22 stitches and 30 rows = 4 inches in After 3 ...


Most gauges are given over 4 inches, with a real minority over 1 or 2 inches, so normalizing matters: 5 stitches over 1 inch is 20 per 4 inches, not 5. About 117 patterns have a gauge but no divisor. I leave those missing rather than assume 4, since a wrong guess would be off by up to 4x. The raw values also run from 0 (a placeholder) to 99.99, which looks like the largest value the form allows and is impossible on a 1-inch divisor.

## 7.2 Put gauge on one scale

I add two columns on the standard scale, stitches and rows per 4 inches (10 cm), and keep the original columns unchanged. Then I set impossible values to missing. The realistic range is my own estimate (roughly super bulky yarn to fine lace and crochet thread), not a Ravelry rule, so I left room on both ends.

In [26]:
# Put stitch and row gauge on one scale: per 4 inches (10 cm). Missing divisor -> NaN
raw_df["sts_per_4in"] = raw_df["gauge"] / raw_df["gauge_divisor"] * 4
raw_df["rows_per_4in"] = raw_df["row_gauge"] / raw_df["gauge_divisor"] * 4

# Realistic range per 4 in: roughly super bulky to fine lace/thread
STS_RANGE = (4, 60)
ROWS_RANGE = (4, 80)

for col, (lo, hi) in [("sts_per_4in", STS_RANGE), ("rows_per_4in", ROWS_RANGE)]:
    bad = raw_df[col].notna() & ~raw_df[col].between(lo, hi)   # includes the 0 placeholders
    print(f"{col}: {bad.sum():,} values outside {lo}-{hi} -> set to missing")
    raw_df.loc[bad, col] = np.nan

raw_df[["sts_per_4in", "rows_per_4in"]].describe()

sts_per_4in: 235 values outside 4-60 -> set to missing
rows_per_4in: 238 values outside 4-80 -> set to missing


,sts_per_4in,rows_per_4in
count,18103.0,15473.0
mean,20.038941,25.004687
std,6.933703,10.69024
min,4.0,4.0
25%,16.0,17.0
50%,20.0,26.0
75%,24.0,32.0
max,60.0,80.0


## 7.3 Does the gauge text agree with the numbers?

`gauge_description` is built from the gauge fields, so it should be present whenever `gauge` is.

In [27]:
# Does gauge_description line up with the gauge columns?
pd.crosstab(raw_df["gauge"].notna(), raw_df["gauge_description"].notna())

gauge_description,False,True
gauge,,
False,10201,1155
True,0,18455


Every pattern with a gauge number has a description. The reverse doesn't hold: about 1,155 patterns have gauge *text* but no number. The most common values show why.

In [28]:
no_num_but_text = raw_df["gauge"].isna() & raw_df["gauge_description"].notna()
raw_df.loc[no_num_but_text, "gauge_description"].value_counts().head(15)

gauge_description
N/A                                         15
US                                          15
Gauge is not important                      13
Not important                               11
see notes below                             10
Gauge not important                         10
gauge is not important                      10
Not Important                                9
see notes                                    9
2 repeats = 4 inches                         8
4 repeats = 4 inches                         8
not important                                8
Gauge is not important for this pattern.     8
No gauge                                     7
single crochet                               7
Name: count, dtype: int64[pyarrow]

## 7.4 Flag patterns where gauge doesn't matter

Most of those are designers saying gauge is not important, spelled a dozen different ways, plus "see notes" and "N/A". These patterns leave gauge unspecified on purpose (blankets, toys, some scarves), so their missing `sts_per_4in` is expected rather than a data gap. I flag them so that difference is available later, for example to check whether gauge-free patterns behave differently.

In [29]:
# Designers who explicitly say gauge doesn't matter (blankets, toys, scarves...)
raw_df["gauge_not_important"] = raw_df["gauge_description"].str.contains(
    r"not (?:important|critical|essential)|no gauge", case=False, na=False
)
print(f"{raw_df['gauge_not_important'].sum():,} patterns say gauge is not important")

raw_df.loc[raw_df["gauge_not_important"], "gauge_description"].value_counts().head(10)

321 patterns say gauge is not important


gauge_description
Gauge is not important                      13
Not important                               11
Gauge not important                         10
gauge is not important                      10
Not Important                                9
not important                                8
Gauge is not important for this pattern.     8
No gauge                                     7
no gauge                                     7
No gauge required                            7
Name: count, dtype: int64[pyarrow]

## 7.5 Yardage

Ravelry records how much yarn a pattern needs as a range: `yardage` is the low end and `yardage_max` is the high end. For example, "100 - 150 yards" is stored as `yardage = 100` and `yardage_max = 150`. When a pattern lists a single amount, only `yardage` is filled in.

A pattern can't need 0 yards, so a 0 means the amount wasn't entered, the same placeholder problem as gauge. I set zeros to missing in two new columns, `yards_min` and `yards_max`, and keep the original columns unchanged.

In [30]:
yard_cols = ["yardage", "yardage_max", "yardage_description"]

print(raw_df[["yardage", "yardage_max"]].describe(), "\n")

print("Zeros:", (raw_df["yardage"] == 0).sum(), "min /", (raw_df["yardage_max"] == 0).sum(), "max")
print("Max present but min missing:", (raw_df["yardage_max"].notna() & raw_df["yardage"].isna()).sum())
print("Max smaller than min:", (raw_df["yardage_max"] < raw_df["yardage"]).sum(), "\n")

# The biggest values: real blankets, or typos?
raw_df.nlargest(15, "yardage")[["name", "type_name"] + yard_cols] if "type_name" in raw_df else raw_df.nlargest(15, "yardage")[["name"] + yard_cols]

           yardage  yardage_max
count      18928.0      15746.0
mean    598.320689   804.311381
std    1000.211927  1275.188974
min            0.0          0.0
25%          153.0        200.0
50%          364.0        440.0
75%         822.75       1148.0
max        88000.0     105600.0 

Zeros: 84 min / 30 max
Max present but min missing: 105
Max smaller than min: 40 



,name,yardage,yardage_max,yardage_description
12330,2303 Dolman Sweater,88000,105600,88000 - 105600 yards
23137,Sophie,40573,51,40573 - 51 yards
26522,S-380 Yellowstone,21000,24000,21000 - 24000 yards
23352,Guest Room,20000,<NA>,20000 yards
5276,Winter’s Garden,18000,21000,18000 - 21000 yards
20060,Rose Splendor,13850,<NA>,13850 yards
24359,Formal Dinner,11260,<NA>,11260 yards
29362,Illinois Modern #S-445,10472,13464,10472 - 13464 yards
22316,Connecticut Modern #S-442,9856,12672,9856 - 12672 yards
24278,The Last Centurion,9000,9700,9000 - 9700 yards


In [31]:
# 0 yards is a placeholder, not a real amount -> missing
raw_df["yards_min"] = raw_df["yardage"].replace(0, pd.NA)
raw_df["yards_max"] = raw_df["yardage_max"].replace(0, pd.NA)

print("Zeros set to missing:", (raw_df["yardage"] == 0).sum(), "min /", (raw_df["yardage_max"] == 0).sum(), "max")
raw_df[["yards_min", "yards_max"]].describe()

Zeros set to missing: 84 min / 30 max


,yards_min,yards_max
count,18844.0,15716.0
mean,600.987795,805.846717
std,1001.639004,1275.920782
min,1.0,1.0
25%,160.0,200.0
50%,367.0,440.0
75%,830.0,1150.0
max,88000.0,105600.0


# Step 8 — Flatten the Nested Columns

Several columns hold a struct (one record per pattern) or a list of structs (several records per pattern). I pull out the parts that are useful for describing and comparing patterns, and Step 9 drops the original nested columns.

## 8.1 Pattern type

`pattern_type` is one struct per pattern with a broad type name (Pullover, Socks, Shawl/Wrap, Home, ...) and a `clothing` flag.

In [32]:
# pattern_type is one struct per pattern: {clothing, id, name, permalink}
raw_df["type_name"] = raw_df["pattern_type"].str.get("name")
raw_df["is_clothing"] = raw_df["pattern_type"].str.get("clothing").astype("boolean")

print(raw_df["is_clothing"].value_counts(dropna=False).to_string(), "\n")
raw_df["type_name"].value_counts(dropna=False)

is_clothing
True     18443
False    11368 



type_name
Toys              3968
Shawl/Wrap        3707
Home              2705
Hat               2446
Pullover          2183
Other             1701
Child             1667
Socks             1620
Scarf             1487
Blanket           1427
Mittens/Gloves    1234
Cardigan          1140
Baby              1083
Bag                870
Dishcloth          541
Tank/Camisole      456
Vest               348
Jacket             327
Tee                272
Dress/Suit         181
Shrug              174
Pet                156
Skirt              107
Naughty             11
Name: count, dtype: int64

## 8.2 Pattern categories

`pattern_categories` is a list, since a pattern can sit in more than one category. Categories are finer-grained than pattern type (for example "Ankle" socks or "Coaster"). I record how many categories each pattern has and take the first listed one as `category_name`. The count shows how often there's more than one.

In [33]:
# Safe length for list-like cells (DuckDB lists can arrive as Python lists or NumPy arrays)
n_items = lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0

# pattern_categories is a list of {id, name, permalink, ...}; take the first listed category
raw_df["n_categories"] = raw_df["pattern_categories"].map(n_items)
raw_df["category_name"] = raw_df["pattern_categories"].str.get(0).str.get("name")

print("Categories per pattern:")
print(raw_df["n_categories"].value_counts().sort_index().to_string(), "\n")

raw_df["category_name"].value_counts().head(20)

Categories per pattern:
n_categories
0       13
1    28564
2     1058
3      131
4       31
5        8
6        4
8        2 



category_name
Pullover                   2628
Beanie, Toque              2190
Shawl / Wrap               1988
Animal                     1755
Other                      1614
Cardigan                   1586
Scarf                      1489
Cowl                       1347
Mid-calf                   1071
Doll                        739
Throw                       703
Fingerless Gloves/Mitts     626
Baby Blanket                562
Sleeveless Top              494
Mittens                     462
Coat / Jacket               445
Vest                        423
Washcloth / Dishcloth       423
Afghan block                399
Dress                       348
Name: count, dtype: int64

## 8.3 Designer

`pattern_author` holds the designer's name and ID plus counts across all of their patterns. Designer popularity is likely one of the biggest influences on a pattern's popularity, so I keep the designer's total pattern count and how many times the designer has been favorited. Both are measured at collection time, across the designer's whole catalog.

In [34]:
# pattern_author struct: designer name/id plus counts across all their patterns
author = raw_df["pattern_author"]
raw_df["author_id"] = author.str.get("id")
raw_df["author_name"] = author.str.get("name")
raw_df["author_patterns_count"] = author.str.get("patterns_count")
raw_df["author_favorites_count"] = author.str.get("favorites_count")

print(f"{raw_df['author_id'].nunique():,} distinct designers\n")
raw_df[["author_patterns_count", "author_favorites_count"]].describe()

14,311 distinct designers



,author_patterns_count,author_favorites_count
count,29811.000000,29811.000000
mean,503.849418,1173.536580
std,1550.732738,3476.533772
min,1.000000,0.000000
25%,24.000000,24.000000
50%,90.000000,169.000000
75%,275.000000,808.500000
max,13550.000000,72508.000000


## 8.4 Pattern attributes

`pattern_attributes` is a list of tags such as "unisex", "in-the-round", or "toddler". I keep the number of tags and the tags themselves as one `|`-separated string, which is easy to search or split later.

In [35]:
# pattern_attributes is a list of {id, permalink} tags (e.g. "unisex", "in-the-round")
raw_df["n_attributes"] = raw_df["pattern_attributes"].map(n_items)
raw_df["attributes"] = raw_df["pattern_attributes"].map(
    lambda attrs: "|".join(a["permalink"] for a in attrs) if n_items(attrs) > 0 else pd.NA
)

print(raw_df["n_attributes"].describe().to_string(), "\n")

# Most common tags across all patterns
raw_df["attributes"].str.split("|").explode().value_counts().head(20)

count    29811.00000
mean         6.39093
std          4.54105
min          0.00000
25%          3.00000
50%          5.00000
75%          9.00000
max         40.00000 



attributes
written-pattern      21403
adult                 9582
in-the-round          9212
worked-flat           9031
female                7382
chart                 7154
one-piece             7024
bottom-up             6440
seamless              5465
seamed                5014
unisex                4330
textured              3982
teen                  3886
phototutorial         3885
stripes-colorwork     3637
lace                  3459
ribbed                3101
top-down              2714
schematic             2381
child                 2363
Name: count, dtype: int64

## 8.5 Languages

`languages` lists every language a pattern is written in. I keep the number of languages, the first listed language, and whether English is one of them, since an English version likely reaches the most Ravelry users.

In [36]:
# languages is a list of {code, id, name, ...}
raw_df["n_languages"] = raw_df["languages"].map(n_items)
raw_df["first_language"] = raw_df["languages"].str.get(0).str.get("name")
raw_df["has_english"] = raw_df["languages"].map(
    lambda langs: n_items(langs) > 0 and any(l["code"] == "en" for l in langs)
)

print(raw_df["has_english"].value_counts().to_string(), "\n")
raw_df["first_language"].value_counts().head(10)

has_english
True     26509
False     3302 



first_language
English      25094
Dutch          723
German         661
Danish         640
Norwegian      414
French         402
Czech          378
Japanese       303
Spanish        194
Finnish        182
Name: count, dtype: int64

## 8.6 Yarn weight

`yarn_weight` is a struct whose name already appears in the flat column `yarn_weight_description` (e.g. "Worsted (9 wpi)"), so I don't need to extract anything. Before using it as a category later, I check its values for placeholders such as "No weight specified".

In [37]:
raw_df["yarn_weight_description"].value_counts(dropna=False)

yarn_weight_description
DK (11 wpi)                           5165
Worsted (9 wpi)                       5050
Aran (8 wpi)                          4577
Fingering (14 wpi)                    4074
Sport (12 wpi)                        2537
<NA>                                  2196
Bulky (7 wpi)                         1777
Super Bulky (5-6 wpi)                 1455
Thread                                1012
Light Fingering                        797
Lace                                   647
Any gauge - designed for any gauge     374
Jumbo (0-4 wpi)                         85
Cobweb                                  32
Aran / Worsted                          21
DK / Sport                              12
Name: count, dtype: int64[pyarrow]

notes is the designer's free-text description of the pattern, written in markdown or HTML and in many languages. I keep the raw text for possible text analysis later, and add two columns: notes_clean, a plain-text copy with HTML tags, links, markdown symbols and extra whitespace removed, and notes_words, its length in words, as a rough measure of how detailed the listing is. Patterns with no description get 0.

In [38]:
# Keep the raw notes; add a plain-text copy for later text analysis
raw_df["notes_clean"] = (
    raw_df["notes"]
    .str.replace(r"<[^>]+>", " ", regex=True)          # HTML tags
    .str.replace(r"https?://\S+", " ", regex=True)     # links
    .str.replace(r"[*_#>`\[\]()]", " ", regex=True)    # markdown symbols
    .str.replace(r"\s+", " ", regex=True)              # line breaks and extra spaces
    .str.strip()
    .replace("", pd.NA)
)

# Description length in words (0 if the designer wrote nothing)
raw_df["notes_words"] = raw_df["notes_clean"].str.split().str.len().fillna(0).astype(int)

raw_df[["notes", "notes_clean", "notes_words"]].dropna().head(3)

,notes,notes_clean,notes_words
0,"Av et nøste Paris, får du litt over 3 brikker....","Av et nøste Paris, får du litt over 3 brikker....",18
1,**Introducing the Ma'alaea Infinity Cowl!**\r\...,Introducing the Ma'alaea Infinity Cowl! Catch ...,55
2,Contrast soled sock in fun sparkly yarn mimics...,Contrast soled sock in fun sparkly yarn mimics...,54


# Step 9 — Drop Columns No Longer Needed

Three groups of columns go:

- **Links, IDs, and account-specific fields.** URLs and product IDs can't predict popularity, `currency_symbol` repeats `currency`, and `pdf_in_library` describes the API user's library rather than the pattern (per the API docs). `yarn_list_type` only says whether the yarn list is exact or a set of suggestions.
- **Nested columns already flattened** in Steps 3, 5, and 8.
- **Nested columns not used in this analysis**: suggested yarns (`packs`), the books or magazines a pattern appears in (`printings`), and needle sizes (`pattern_needle_sizes`, which mostly repeat what gauge and yarn weight already capture).

In [39]:
LINKS_AND_IDS = [
    "url", "pdf_url", "permalink",      # links
    "product_id",                       # ID of the "buy now" product listing
    "currency_symbol",                  # redundant with currency
    "pdf_in_library",                   # about the API user's library, not the pattern
    "yarn_list_type",                   # whether the yarn list is exact or suggestions; not used
]
NESTED_FLATTENED = [
    "craft",                            # -> craft_name (Step 3)
    "download_location",                # -> download_type (Step 5)
    "pattern_type",                     # -> type_name, is_clothing (Step 8)
    "pattern_categories",               # -> n_categories, category_name
    "pattern_author",                   # -> author_* columns
    "pattern_attributes",               # -> n_attributes, attributes
    "languages",                        # -> n_languages, first_language, has_english
    "yarn_weight",                      # -> yarn_weight_description already flat
]
NESTED_NOT_USED = ["packs", "printings", "pattern_needle_sizes"]

raw_df = raw_df.drop(columns=LINKS_AND_IDS + NESTED_FLATTENED + NESTED_NOT_USED)
raw_df.shape

(29811, 58)

# Step 10 — Final Check and Save

The cleaned table has one row per knitting or crochet pattern, with placeholders turned into real missing values, prices in USD, time on Ravelry measured, gauge on a single scale, and the nested fields flattened. A last look at the missing values and a few rows:

In [40]:
clean_df = raw_df.reset_index(drop=True)

print(clean_df.shape)
clean_df.isna().sum()

(29811, 58)


comments_count                 0
created_at                     0
currency                    5720
difficulty_average         22193
difficulty_count               0
downloadable                   0
favorites_count                0
free                           0
gauge                      11356
gauge_divisor               4154
gauge_pattern              15164
generally_available         1289
id                             0
name                           0
price                      15310
projects_count                 0
published                   5345
queued_projects_count          0
rating_average             22242
rating_count                   0
row_gauge                  14010
updated_at                     0
yardage                    10883
yardage_max                14065
sizes_available             8387
ravelry_download               0
gauge_description          10201
yarn_weight_description     2196
yardage_description            0
notes                       3321
craft_name

In [41]:
clean_df.head(3)

,comments_count,created_at,currency,difficulty_average,difficulty_count,downloadable,favorites_count,free,gauge,gauge_divisor,gauge_pattern,generally_available,id,name,price,projects_count,published,queued_projects_count,rating_average,rating_count,row_gauge,updated_at,yardage,yardage_max,sizes_available,ravelry_download,gauge_description,yarn_weight_description,yardage_description,notes,craft_name,availability,download_type,price_usd,created_date,published_date,available_date,years_on_ravelry,sts_per_4in,rows_per_4in,gauge_not_important,yards_min,yards_max,type_name,is_clothing,n_categories,category_name,author_id,author_name,author_patterns_count,author_favorites_count,n_attributes,attributes,n_languages,first_language,has_english,notes_clean,notes_words
0,0,2014/04/05 19:17:30 -0400,NOK,NaN,0,True,46,False,NaN,4,<NA>,2014/04/05 19:17:30 -0400,483075,Glassbrikke àla Mon Ami!,20.0,1,2014/04/01,0,NaN,0,NaN,2014/04/05 19:27:08 -0400,<NA>,<NA>,<NA>,True,<NA>,Aran (8 wpi),yards,"Av et nøste Paris, får du litt over 3 brikker....",Crochet,paid_download,ravelry,2.04,2014-04-05,2014-04-01,2014-04-05,12.635181,<NA>,<NA>,False,<NA>,<NA>,Home,False,1,Coaster,57731,Mon Ami Design,10,3,1,in-the-round,1,Norwegian,False,"Av et nøste Paris, får du litt over 3 brikker....",18
1,0,2020/12/16 15:16:08 -0500,USD,2.8,5,True,47,False,36.0,4,Garter Stitch,2020/12/16 15:16:08 -0500,1096269,Ma'alaea,9.5,10,2020/12/01,4,4.6,5,32.0,2025/10/31 11:24:11 -0400,<NA>,<NA>,"Circumference: 56""; Height: 7""; easily adjustable",True,36 stitches and 32 rows = 4 inches in Garter S...,Worsted (9 wpi),yards,**Introducing the Ma'alaea Infinity Cowl!**\r\...,Knitting,paid_download,ravelry,9.5,2020-12-16,2020-12-01,2020-12-16,5.935661,36.0,32.0,False,<NA>,<NA>,Shawl/Wrap,True,1,Cowl,733,Melissa Leapman,1519,2400,15,unisex|cables|textured|kitchener|one-piece|sho...,1,English,True,Introducing the Ma'alaea Infinity Cowl! Catch ...,55
2,1,2013/03/28 20:16:50 -0400,USD,NaN,0,False,7,False,32.0,4,Garter abd Rib stitch pattern,2013/06/01 00:00:00 -0400,395239,Cirque du Sole,NaN,2,2013/06/01,1,NaN,0,36.0,2013/08/09 00:15:10 -0400,100,150,Childs S-M-L,False,32 stitches and 36 rows = 4 inches in Garter a...,DK (11 wpi),100 - 150 yards,Contrast soled sock in fun sparkly yarn mimics...,Knitting,print_only,<NA>,<NA>,2013-03-28,2013-06-01,2013-06-01,13.656400,32.0,36.0,False,100,150,Socks,True,1,Ankle,19689,Alice Curtis,38,76,6,toddler|top-cuff-down|heel-flap|seamed-sock|In...,1,English,True,Contrast soled sock in fun sparkly yarn mimics...,54


I save the cleaned table to the same DuckDB database as the raw table, so the EDA page can load it with one query.

In [42]:
# DuckDB can read a pandas DataFrame directly by its variable name
con.execute("CREATE OR REPLACE TABLE clean_patterns AS SELECT * FROM clean_df")
print(con.execute("SELECT COUNT(*) AS n_patterns FROM clean_patterns").fetchone())

(29811,)
